# Gather Generator HPO Experiment Results
Use this notebook to gather results form the hyperparameter optimization pipeline.

In [ ]:
import pandas as pd
import os

# Define the main folder
main_folder = "../experiments/generators_hpo/TabDDPM_census/"

# List all subdirectories that are numbers
subfolders = sorted([f for f in os.listdir(main_folder) if f.isdigit()], key=int)

# Read CSV files and store DataFrames in a list
df_list = []
for subfolder in subfolders:
    csv_path = os.path.join(main_folder, subfolder, "log.csv")  # Adjust the filename if necessary
    if os.path.exists(csv_path):  # Check if the CSV file exists
        df = pd.read_csv(csv_path)
        df["source_folder"] = subfolder  # Optional: Add a column to track source folder
        df_list.append(df)

# Concatenate all DataFrames
if df_list:
    final_df = pd.concat(df_list, ignore_index=True)
    # define source_folder as index
    final_df = final_df.rename(columns={'source_folder': 'exp_number'})
    final_df['exp_number'] = final_df['exp_number'].astype(int)
    final_df = final_df.set_index('exp_number')
else:
    print("No CSV files found.")

final_df = final_df[['score_large', 'score_authenticity', 'score_difference', 'score_small', 'std_small']].sort_values(by='score_large', ascending=False).round(3)
final_df

In [ ]:
import numpy as np
utilities = [0.6, 0.65, 0.7, 0.75, 0.8, 0.75]
privacies = [0.8, 0.75, 0.7, 0.65, 0.6, 0.7]
weightings = [0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

df_list = []
for utility, privacy in zip(utilities, privacies):
    row = {'utility': utility, 'privacy': privacy}
    for weight in weightings:
        row[f'privacy_{weight}'] = utility*(1-weight) + privacy*weight
    df_list.append(row)

df = pd.DataFrame(df_list)
df